# 01 — Data Inspection & Cleaning

**Phase 1 scope:** quality audit, nulls, duplicates, inconsistent values, key validation and reproducible cleaning. This notebook uses real Instacart sampled order-level data bundled in data/raw/.

**No modeling is performed.**

**Data mode:** `real` uses observed Instacart transactions and is only for testing. Set `DATA_MODE = 'real'` and rerun all cells after placing original Kaggle CSVs in `data/raw/`.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# Analysis on the original Instacart CSVs; a reproducible sample of 30,000 prior orders.
DATA_MODE = 'real'
print('DATA MODE:', DATA_MODE.upper(), '|', 'REAL INSTACART SAMPLE' if DATA_MODE == 'demo' else 'ORIGINAL INSTACART DATA')

DATA MODE: REAL | ORIGINAL INSTACART DATA


## Load source tables and inspect schema
We sample prior **orders** reproducibly before joining the potentially huge line-items file. This avoids silently treating a random selection of line items as complete baskets.

In [2]:
import pandas as pd, json
from IPython.display import display
processed = ROOT/'data/processed'
report = json.loads((processed/'cleaning_report.json').read_text())
print('Original prior order count:', report['total_prior_orders_in_source'])
for name, audit_data in report['audit_before'].items():
    print(name, 'rows:', audit_data['rows'], 'missing:', audit_data['missing_by_column'], 'duplicates:', audit_data['exact_duplicates'])
print('Sampling method:', report['sampling_method'])

Original prior order count: 3214874
orders rows: 30000 missing: {'order_id': 0, 'user_id': 0, 'eval_set': 0, 'order_number': 0, 'order_dow': 0, 'order_hour_of_day': 0, 'days_since_prior_order': 1894} duplicates: 0
order_products rows: 302305 missing: {'order_id': 0, 'product_id': 0, 'add_to_cart_order': 0, 'reordered': 0} duplicates: 0
products rows: 49688 missing: {'product_id': 0, 'product_name': 0, 'aisle_id': 0, 'department_id': 0} duplicates: 0
aisles rows: 134 missing: {'aisle_id': 0, 'aisle': 0} duplicates: 0
departments rows: 21 missing: {'department_id': 0, 'department': 0} duplicates: 0
Sampling method: deterministic random sample of prior order IDs with seed 42; item lines filtered in chunks; not a time series split


## Apply documented cleaning
- Remove exact and key-based duplicates; remove rows without critical IDs.
- Validate time and item-position ranges and catalog foreign keys.
- Keep first-order `days_since_prior_order` nulls (structurally expected).
- Do not remove valid large baskets as “outliers” automatically.
- Generate `cleaning_report.json` containing before/after audit counts.

In [3]:
from scripts.pipeline import clean_and_build, load_data, run
# Reproducible cleaning implementation lives in scripts/pipeline.py.
# Run `python -m scripts.run_phase1 --max-orders 30000` to regenerate these tables.
print('Cleaning decisions:',report['cleaning'])
print('After cleaning:',report['after'])
outputs = {name: pd.read_csv(processed/f'{name}.csv', nrows=10) for name in ['orders','product_catalog','order_products_clean','baskets']}

Cleaning decisions: {'removed_duplicate_or_null_key_rows': {'order_id': 0, 'order_id,product_id': 0, 'product_id': 0, 'aisle_id': 0, 'department_id': 0}, 'invalid_order_time_rows_removed': 0, 'invalid_line_flags_or_positions_removed': 0, 'first_order_missing_days_since_prior_order': 'kept as NA (expected structural missingness)', 'outlier_policy': 'invalid hours/days/positions excluded; true high basket sizes retained; 0..30 days valid interval', 'scaling': 'Not applied: categorical product IDs are identifiers and Apriori/FP-Growth consume binary basket presence, not continuous features.'}
After cleaning: {'orders': 30000, 'order_product_rows': 302305, 'products_catalog': 49688, 'nonempty_baskets': 30000}


## Inspect the cleaned outputs
These tables support future basket-based mining, but neither recommendations nor rule models are trained in this phase.

In [4]:
display(outputs['product_catalog'].head());display(outputs['baskets'].head())

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


,order_id,user_id,basket_size,product_ids,product_names,order_dow,order_hour_of_day
0,136,184355,4,11005|18465|29926|47766,Organic Avocado | Organic Cream Cheese Bar | O...,6,10
1,376,121858,7,890|2716|8277|16462|35851|45210|47029,Apple Honeycrisp Organic | Organic Diced Tomat...,5,16
2,522,35250,20,1862|2344|10251|13292|17706|24184|24852|25340|...,Africa Single Origin 1 Coffee | Banana | Cryst...,1,15
3,526,17900,18,1752|1831|1940|9387|12191|12302|22489|23085|27...,"Beef Franks | Country Stand Juice, Medium Pulp...",0,23
4,552,18926,4,4091|5268|33452|37011,Artichokes | Bagel Bites Three Cheese Mini Bag...,4,13
